# ⚙️ Fine-Tuning Hyperparameters

**Optimize hyperparameters for better fine-tuned models**

---

## 📋 Overview

**What you'll learn:**
- Key hyperparameters explained
- Learning rate selection
- Batch size and epochs
- Hyperparameter tuning strategies
- Best practices and defaults

**Time estimate:** ⏱️ 45 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List
import pandas as pd

print("✅ Setup complete")

## 🤔 Why Hyperparameters Matter

### Same Data, Different Results:

```
Model A (bad hyperparams):
  Epochs: 10
  Learning rate: 0.1
  → Accuracy: 60% (overfitting!)

Model B (good hyperparams):
  Epochs: 3
  Learning rate: 0.0003
  → Accuracy: 92% ✅
```

### Key Hyperparameters:

1. **Learning Rate** - How fast the model learns
2. **Epochs** - How many times to see the data
3. **Batch Size** - How many examples per update
4. **Learning Rate Schedule** - How to adjust LR over time
5. **Warmup Steps** - Gradual learning rate increase

### Impact on Training:

| Setting | Too Low | Too High |
|---------|---------|----------|
| **Learning Rate** | Slow convergence | Unstable, diverges |
| **Epochs** | Underfitting | Overfitting |
| **Batch Size** | Slow, noisy | Fast but less generalization |

## 📚 Learning Rate

In [ ]:
# Simulate learning with different learning rates
def simulate_learning(learning_rate: float, steps: int = 100) -> List[float]:
    """Simulate training loss with given learning rate."""
    
    loss = 2.0  # Starting loss
    losses = []
    
    for step in range(steps):
        # Simulated gradient
        gradient = -0.1 * loss + np.random.normal(0, 0.1)
        
        # Update
        loss = loss - learning_rate * gradient
        
        # Add noise
        loss = max(0.1, loss + np.random.normal(0, 0.05))
        
        losses.append(loss)
    
    return losses

# Test different learning rates
learning_rates = [0.001, 0.01, 0.1, 0.5]

plt.figure(figsize=(12, 6))

for lr in learning_rates:
    losses = simulate_learning(lr)
    plt.plot(losses, label=f'LR = {lr}')

plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('Impact of Learning Rate on Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/home/user/Rag_notebooks/06_fine_tuning/learning_rate_comparison.png', dpi=100, bbox_inches='tight')
print("📊 Saved learning rate comparison plot")

print("\n📈 Learning Rate Guidelines:")
print("")
print("For OpenAI fine-tuning:")
print("  • Default: Auto (recommended)")
print("  • Range: 0.00001 - 0.001")
print("  • Typical: 0.0003")
print("")
print("For LoRA/PEFT:")
print("  • Range: 0.0001 - 0.0005")
print("  • Typical: 0.0003")
print("")
print("💡 Start with default, adjust only if needed!")

## 🔄 Number of Epochs

In [ ]:
def simulate_epochs(num_epochs: int) -> Dict:
    """Simulate training and validation loss over epochs."""
    
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        # Training loss decreases
        train_loss = 2.0 * np.exp(-0.3 * epoch) + np.random.normal(0, 0.05)
        
        # Validation starts to increase after overfitting
        if epoch < 5:
            val_loss = 2.0 * np.exp(-0.25 * epoch) + np.random.normal(0, 0.1)
        else:
            # Overfitting - val loss increases
            val_loss = 0.5 + 0.1 * (epoch - 5) + np.random.normal(0, 0.1)
        
        train_losses.append(max(0.1, train_loss))
        val_losses.append(max(0.1, val_loss))
    
    return {
        'train_losses': train_losses,
        'val_losses': val_losses
    }

# Simulate
results = simulate_epochs(10)

plt.figure(figsize=(10, 6))
plt.plot(results['train_losses'], 'b-', label='Training Loss', linewidth=2)
plt.plot(results['val_losses'], 'r-', label='Validation Loss', linewidth=2)
plt.axvline(x=4, color='g', linestyle='--', label='Optimal stopping point', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Over Epochs')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/home/user/Rag_notebooks/06_fine_tuning/epochs_comparison.png', dpi=100, bbox_inches='tight')
print("📊 Saved epochs comparison plot")

print("\n🔄 Epoch Guidelines:")
print("")
print("General recommendations:")
print("  • Small dataset (< 100):   1-2 epochs")
print("  • Medium (100-1000):       2-4 epochs")
print("  • Large (1000+):           3-5 epochs")
print("")
print("⚠️  Warning signs:")
print("  • Train loss ↓, Val loss ↑ → Overfitting! Reduce epochs")
print("  • Both losses still high → Underfitting! Increase epochs")
print("")
print("💡 Use early stopping if available!")

## 📦 Batch Size

In [ ]:
# Batch size comparison
batch_comparison = pd.DataFrame([
    {
        'Batch Size': 1,
        'Speed': 'Slowest',
        'Memory': 'Lowest',
        'Gradient': 'Noisy',
        'Generalization': 'Better',
        'Use When': 'Small GPU',
    },
    {
        'Batch Size': 4,
        'Speed': 'Slow',
        'Memory': 'Low',
        'Gradient': 'Moderate',
        'Generalization': 'Good',
        'Use When': 'Default choice',
    },
    {
        'Batch Size': 8,
        'Speed': 'Medium',
        'Memory': 'Medium',
        'Gradient': 'Stable',
        'Generalization': 'Good',
        'Use When': 'Recommended',
    },
    {
        'Batch Size': 32,
        'Speed': 'Fast',
        'Memory': 'High',
        'Gradient': 'Very stable',
        'Generalization': 'Worse',
        'Use When': 'Large GPU, large dataset',
    },
])

print("📦 Batch Size Comparison\n")
print(batch_comparison.to_string(index=False))

print("\n💡 Recommendations:")
print("")
print("OpenAI Fine-Tuning:")
print("  • Use 'auto' (recommended)")
print("  • Manual: 1, 2, 4, 8")
print("")
print("Local Fine-Tuning:")
print("  • Small GPU (8GB):  batch_size=1-2")
print("  • Medium (16GB):    batch_size=4-8")
print("  • Large (40GB+):    batch_size=16-32")
print("")
print("💡 Larger batch = faster but needs more memory")

## 🎯 Hyperparameter Tuning Strategy

In [ ]:
class HyperparameterTuner:
    """Simple hyperparameter tuning helper."""
    
    @staticmethod
    def recommend_hyperparams(
        dataset_size: int,
        task_type: str = "general"
    ) -> Dict:
        """Recommend hyperparameters based on dataset."""
        
        if dataset_size < 100:
            return {
                'n_epochs': 1,
                'batch_size': 'auto',
                'learning_rate_multiplier': 2.0,
                'reasoning': 'Small dataset - risk of overfitting'
            }
        elif dataset_size < 500:
            return {
                'n_epochs': 2,
                'batch_size': 'auto',
                'learning_rate_multiplier': 1.5,
                'reasoning': 'Medium dataset - balanced approach'
            }
        elif dataset_size < 2000:
            return {
                'n_epochs': 3,
                'batch_size': 'auto',
                'learning_rate_multiplier': 1.0,
                'reasoning': 'Large dataset - default settings'
            }
        else:
            return {
                'n_epochs': 4,
                'batch_size': 'auto',
                'learning_rate_multiplier': 0.5,
                'reasoning': 'Very large dataset - conservative LR'
            }
    
    @staticmethod
    def adjust_for_results(
        current_hyperparams: Dict,
        train_loss: float,
        val_loss: float
    ) -> Dict:
        """Suggest adjustments based on results."""
        
        suggestions = []
        new_hyperparams = current_hyperparams.copy()
        
        # Check for overfitting
        if val_loss > train_loss * 1.2:
            suggestions.append("⚠️  Overfitting detected")
            suggestions.append("  → Reduce epochs")
            suggestions.append("  → Add more data if possible")
            
            new_hyperparams['n_epochs'] = max(1, current_hyperparams.get('n_epochs', 3) - 1)
        
        # Check for underfitting
        elif train_loss > 0.5:
            suggestions.append("⚠️  High training loss")
            suggestions.append("  → Increase epochs")
            suggestions.append("  → Increase learning rate")
            
            new_hyperparams['n_epochs'] = current_hyperparams.get('n_epochs', 3) + 1
            new_hyperparams['learning_rate_multiplier'] = current_hyperparams.get('learning_rate_multiplier', 1.0) * 1.5
        
        else:
            suggestions.append("✅ Training looks good!")
        
        return {
            'suggestions': suggestions,
            'new_hyperparams': new_hyperparams
        }

# Example usage
print("🎯 Hyperparameter Recommendations\n")
print("="*60)

dataset_sizes = [50, 250, 1000, 5000]

for size in dataset_sizes:
    rec = HyperparameterTuner.recommend_hyperparams(size)
    print(f"\nDataset size: {size}")
    print(f"  Epochs: {rec['n_epochs']}")
    print(f"  Batch size: {rec['batch_size']}")
    print(f"  LR multiplier: {rec['learning_rate_multiplier']}")
    print(f"  Reasoning: {rec['reasoning']}")

# Example adjustment
print("\n\n🔧 Adjusting Based on Results\n")
print("="*60)

scenarios = [
    {"train": 0.1, "val": 0.8, "desc": "Overfitting"},
    {"train": 0.8, "val": 0.85, "desc": "Underfitting"},
    {"train": 0.2, "val": 0.22, "desc": "Good fit"},
]

for scenario in scenarios:
    print(f"\nScenario: {scenario['desc']}")
    print(f"  Train loss: {scenario['train']}")
    print(f"  Val loss: {scenario['val']}")
    
    result = HyperparameterTuner.adjust_for_results(
        {'n_epochs': 3, 'learning_rate_multiplier': 1.0},
        scenario['train'],
        scenario['val']
    )
    
    print("  Suggestions:")
    for suggestion in result['suggestions']:
        print(f"    {suggestion}")

## 📋 Complete Hyperparameter Guide

In [ ]:
# Complete guide
hyperparameter_guide = {
    "Learning Rate": {
        "Description": "How fast the model learns",
        "OpenAI Default": "auto",
        "Manual Range": "0.00001 - 0.001",
        "Recommended": "0.0003",
        "Too Low": "Slow learning, may not converge",
        "Too High": "Unstable, may diverge",
    },
    "Epochs": {
        "Description": "Number of passes through data",
        "OpenAI Default": "auto",
        "Manual Range": "1 - 10",
        "Recommended": "2-4",
        "Too Low": "Underfitting",
        "Too High": "Overfitting",
    },
    "Batch Size": {
        "Description": "Examples per gradient update",
        "OpenAI Default": "auto",
        "Manual Range": "1, 2, 4, 8",
        "Recommended": "4 or auto",
        "Too Low": "Slow, noisy gradients",
        "Too High": "Fast but worse generalization",
    },
}

print("📋 Complete Hyperparameter Guide\n")
print("="*70)

for param, details in hyperparameter_guide.items():
    print(f"\n{param}:")
    for key, value in details.items():
        print(f"  {key:20} {value}")

## ✅ Summary

### Key Hyperparameters:

**1. Learning Rate**
```python
# OpenAI
learning_rate_multiplier: "auto"  # Recommended!
# Or manually: 0.5, 1.0, 1.5, 2.0

# Local training
learning_rate: 3e-4  # Default for LoRA
```

**2. Epochs**
```python
# Dataset size → Epochs
< 100:     1-2 epochs
100-1000:  2-4 epochs
1000+:     3-5 epochs

# Rule: Start low, increase if underfitting
```

**3. Batch Size**
```python
# OpenAI: Use "auto"
batch_size: "auto"

# Manual:
Small GPU:  1-2
Medium:     4-8
Large:      16-32
```

### Quick Start Configurations:

**Small Dataset (< 100 examples):**
```python
{
  "n_epochs": 1,
  "batch_size": "auto",
  "learning_rate_multiplier": 2.0
}
```

**Medium Dataset (100-1000):**
```python
{
  "n_epochs": 3,
  "batch_size": "auto",
  "learning_rate_multiplier": "auto"
}
```

**Large Dataset (1000+):**
```python
{
  "n_epochs": 4,
  "batch_size": "auto",
  "learning_rate_multiplier": 1.0
}
```

### Tuning Strategy:

**Step 1: Start with defaults**
```python
# Let OpenAI optimize
hyperparameters = {
    "n_epochs": "auto",
    "batch_size": "auto",
    "learning_rate_multiplier": "auto"
}
```

**Step 2: Check results**
```python
if overfitting:
    epochs -= 1
    
if underfitting:
    epochs += 1
    learning_rate *= 1.5
```

**Step 3: Fine-tune**
```python
# Only adjust if needed!
# Small changes: 0.5x - 2x
# Test on validation set
```

### Common Issues & Solutions:

**Overfitting (train good, val bad):**
```python
✓ Reduce epochs
✓ Add more training data
✓ Reduce learning rate
✓ Enable early stopping
```

**Underfitting (both train & val bad):**
```python
✓ Increase epochs
✓ Increase learning rate
✓ Check data quality
✓ Increase model size (LoRA rank)
```

**Unstable training (loss jumps):**
```python
✓ Reduce learning rate (halve it)
✓ Increase batch size
✓ Add gradient clipping
✓ Check for bad data
```

**Training too slow:**
```python
✓ Increase batch size
✓ Increase learning rate
✓ Use gradient accumulation
✓ Use mixed precision training
```

### Advanced Techniques:

**1. Learning Rate Scheduling**
```python
# Cosine schedule
lr_scheduler_type="cosine"

# Linear warmup
warmup_steps=100
```

**2. Gradient Accumulation**
```python
# Effective batch size = batch_size × accumulation_steps
per_device_train_batch_size=2
gradient_accumulation_steps=4
# Effective batch size = 8
```

**3. Early Stopping**
```python
# Stop if val loss doesn't improve
early_stopping_patience=2
```

### Best Practices:

1. **Always use validation set**
   - Monitor for overfitting
   - Don't tune on test set!

2. **Start simple**
   - Use "auto" for OpenAI
   - Use defaults for local
   - Only tune if results are bad

3. **One change at a time**
   - Change one hyperparameter
   - Measure impact
   - Repeat

4. **Track everything**
   ```python
   experiment_log = {
       'hyperparams': {...},
       'train_loss': 0.2,
       'val_loss': 0.25,
       'notes': 'Reduced LR, improved val loss'
   }
   ```

### Decision Tree:

```
First run complete?
├─ Train & val both high (> 0.5)
│  └─ Underfitting → Increase epochs or LR
├─ Train low, val high
│  └─ Overfitting → Reduce epochs, add data
└─ Both low (< 0.3)
   └─ Good! → Maybe slight tuning
```

### Recommended Defaults:

**OpenAI API:**
```python
# Best starting point
hyperparameters={
    "n_epochs": "auto",
    "batch_size": "auto",
    "learning_rate_multiplier": "auto"
}
```

**Local LoRA:**
```python
# Good defaults
training_args = TrainingArguments(
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
)
```

### Next Steps:

You've completed the Fine-Tuning module!

**Next module:** `07_agents_tools/` - Building AI agents with tools